# Kavra TTS sunucusu (Colab)

Bu not defteri, GPU'su olmayan Kavra sunucunun ses üretimini (**XTTS v2** ve **Piper**) Colab GPU'suna devretmesini sağlar.

1. **Çalışma zamanı → Çalışma zamanı türünü değiştir → GPU** seç.
2. Aşağıdaki hücreleri sırayla çalıştır.
3. Son hücrenin yazdırdığı **Adres** ve **Token**'ı Kavra'da *Ses ayarları → Çalıştırma yeri: Uzak GPU* alanlarına gir.
4. Render sürerken **son hücreyi ve bu sekmeyi açık bırak.**

*İsteğe bağlı:* Token'ın her oturumda aynı kalması için Colab'ın sol menüsündeki **Secrets (🔑)** bölümüne `KAVRA_TTS_TOKEN` adıyla en az 16 karakterlik bir değer ekle.

> Not: Colab oturumu kapanır veya adres yenilenirse Kavra'da yeni adresi girip renderı yeniden başlat; biten sesler korunur.

In [ ]:
# 1) GPU kontrolü ve paketi yükle
import os, shutil, subprocess, zipfile
from google.colab import files

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "YOK - Çalışma zamanı türünü GPU yap ve hücreyi yeniden çalıştır")

print("kavra-tts-bundle.zip dosyasını seç:")
uploaded = files.upload()
archive = next((name for name in uploaded if name.endswith(".zip")), None)
assert archive, "zip dosyası seçilmedi"
shutil.rmtree("/content/kavra", ignore_errors=True)
zipfile.ZipFile(archive).extractall("/content/kavra")
print("Paket açıldı:", sorted(os.listdir("/content/kavra")))

In [ ]:
# 2) Bağımlılıkları kur (birkaç dakika sürer)
import subprocess, sys

result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/kavra/tts_server/requirements.txt"])
assert result.returncode == 0, "Kurulum başarısız - yukarıdaki pip hatasını kontrol et"
print("Kurulum tamam")

In [ ]:
# 3) Sunucuyu başlat (modeller GPU'ya yüklenir, ~1-2 dk)
import os, secrets, subprocess, sys, time, requests

try:
    from google.colab import userdata
    TOKEN = userdata.get("KAVRA_TTS_TOKEN")
except Exception:
    TOKEN = None
TOKEN = TOKEN or secrets.token_urlsafe(24)

PORT = 8790
COQUI_MODELS = 2  # aynı anda yüklü XTTS kopyası; T4 (16 GB) için 2 uygundur, bellek yetmezse 1 yap

log = open("/content/kavra_tts.log", "w")
server = subprocess.Popen(
    [sys.executable, "-m", "tts_server.server", "--port", str(PORT), "--coqui-models", str(COQUI_MODELS),
     "--data-dir", "/content/kavra_tts_data"],
    cwd="/content/kavra", env={**os.environ, "KAVRA_TTS_TOKEN": TOKEN}, stdout=log, stderr=subprocess.STDOUT)

info = None
for _ in range(300):
    if server.poll() is not None:
        break
    try:
        info = requests.get(f"http://127.0.0.1:{PORT}/health", headers={"Authorization": f"Bearer {TOKEN}"}, timeout=2).json()
        break
    except Exception:
        time.sleep(2)
if info is None:
    print(open("/content/kavra_tts.log").read()[-3000:])
    raise SystemExit("Sunucu başlamadı - yukarıdaki günlüğe bak")
print("GPU:", info["gpu"])
for name, state in info["engines"].items():
    print(f"  {name}: {'hazır' if state['available'] else 'KULLANILAMIYOR - ' + str(state['error'])}")

In [ ]:
# 4) İnternete aç (cloudflared, hesap gerekmez) ve bağlantı bilgilerini yazdır
import sys
sys.path.insert(0, "/content/kavra")
from tts_server.colab_tunnel import start_tunnel

tunnel, URL = start_tunnel(PORT)

def show():
    print("=" * 62)
    print("Kavra > Ses ayarları > Uzak GPU alanlarına gir:")
    print("  Adres:", URL)
    print("  Token:", TOKEN)
    print("=" * 62)
show()

In [ ]:
# 5) Bu hücreyi açık bırak: sunucu ve tüneli izler. Durdurmak için ■ düğmesine bas.
import time

try:
    while True:
        time.sleep(300)
        server_ok = server.poll() is None
        print(time.strftime("%H:%M"), "| sunucu:", "çalışıyor" if server_ok else "KAPANDI - 3. hücreyi yeniden çalıştır")
        if tunnel.poll() is not None:
            tunnel, URL = start_tunnel(PORT)
            print("Tünel yenilendi, ADRES DEĞİŞTİ:")
            show()
except KeyboardInterrupt:
    tunnel.terminate(); server.terminate()
    print("Durduruldu.")